# OLX Market Scanner — Straż Przyszłości

## Architektura 3(+1)-fazowa
1. **FAZA 1 — Pobierz wszystko**: Scrape ALL offers, minimize API calls
2. **FAZA 2 — Kataloguj lokalnie**: Tag + categorize in SQLite (keyword)
3. **FAZA 2B — AI Tagging** *(opcjonalnie)*: Gemma 4 na GPU Kaggle
4. **FAZA 3 — Eksportuj na GitHub**: Push JSONL + DB + raport

## Watchlisty Krzyska
- 🏠 **Wynajem**: dom/mieszkanie max 20km Kłodzko, do 1500 zł
- 🔌 **Darmowa elektronika**: 20km Kłodzko
- 🧊 **Darmowe AGD**: 20km Kłodzko

### Zasady:
- Najpierw pobierz, potem analizuj — zero AI calls podczas scrapowania
- Oszczędzaj limity — baza na GitHub, D1 na przyszłość
- Gemma 4 AI tagging opcjonalny — `GEMMA_ENABLED = True` aby włączyć


In [ ]:
# Cell 1: Install dependencies
!pip install -q httpx 2>/dev/null
# Gemma 4 deps (uncomment when enabling AI tagging):
# !pip install -q kagglehub transformers accelerate bitsandbytes 2>/dev/null

import json, hashlib, time, os, re, subprocess, socket
from datetime import datetime, timezone
from pathlib import Path
import sqlite3, httpx
print(f'✅ Ready. httpx={httpx.__version__}')


In [ ]:
# Configuration — watchlists + targets
OLX_BASE_URL = 'https://www.olx.pl/api/v1/offers'
OLX_BROWSER_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.olx.pl/',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'same-origin',
}
OLX_HEADERS = {
    **OLX_BROWSER_HEADERS,
    'Accept': 'application/json, text/plain, */*',
    'Origin': 'https://www.olx.pl',
}
OLX_BLOCK_STATUS_CODES = {403, 429}
MAX_FETCH_ATTEMPTS = 3

CAT_FREE = 1151; CAT_ELECTRONICS = 99; CAT_HOME = 506
CAT_RENT_FLAT = 1311; CAT_RENT_HOUSE = 1313

WATCHLISTS = {
    'wynajem': {
        'categories': [CAT_RENT_FLAT, CAT_RENT_HOUSE],
        'city_id': 47303, 'distance': 20, 'max_price': 1500,
        'label': '🏠 Wynajem do 1500zł',
    },
    'darmowa_elektronika': {
        'categories': [CAT_ELECTRONICS],
        'city_id': 47303, 'distance': 20, 'price_free': True,
        'label': '🔌 Darmowa elektronika',
    },
    'darmowe_agd': {
        'categories': [CAT_HOME],
        'city_id': 47303, 'distance': 20, 'price_free': True,
        'label': '🧊 Darmowe AGD',
    },
}

FREE_SCAN_TARGETS = [
    ('Kłodzko',47303,50), ('Wałbrzych',47174,30),
    ('Dzierżoniów',46639,30), ('Nysa',48317,30), ('Wrocław',47393,20),
]

REQUEST_DELAY_SEC = 1.2; PAGE_LIMIT = 50; MAX_PAGES = 25
OUTPUT_DIR = Path('/kaggle/working/olx_scraper')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = OUTPUT_DIR / 'olx_market.db'
WATCHLIST_DIR = OUTPUT_DIR / 'watchlists'
WATCHLIST_DIR.mkdir(exist_ok=True)
JSONL_PATH = OUTPUT_DIR / 'olx_offers_export.jsonl'
SQL_PATH = OUTPUT_DIR / 'olx_d1_import.sql'
REPORT_PATH = OUTPUT_DIR / 'olx_report.md'
GITHUB_PAT = os.environ.get('GITHUB_PAT', '')
GITHUB_REPO = os.environ.get('GITHUB_REPO', 'StrazPrzyszlosci/STRAZ_PRZYSZLOSCI')
GITHUB_BRANCH = 'main'
GITHUB_DATA_PATH = 'PROJEKTY/13_baza_czesci_recykling/olx_data'
SCRAPER_VERSION = '2.2.0'
SCAN_BATCH_ID = datetime.now(timezone.utc).strftime('olx-scan-%Y%m%d-%H%M%S')
print(f'✅ Config. Batch: {SCAN_BATCH_ID}')

# Gemma 4 AI config (used only when GEMMA_ENABLED=True in FAZA 2B)
GEMMA_ENABLED = False  # Master switch — also set in FAZA 2B cell
GEMMA_MODEL_VARIANT = 'gemma-4-26b-a4b-it'  # Best on P100
GEMMA_QUANTIZATION = '4bit'  # '4bit' or 'full'
GEMMA_BATCH_LIMIT = 200  # Max offers per AI run (GPU quota: 30h/week)
print(f'   Gemma AI: {"✅ ON (" + GEMMA_MODEL_VARIANT + ")" if GEMMA_ENABLED else "⏭️ OFF"}')


In [ ]:
# SQLite database
def init_db(db_path):
    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA foreign_keys=ON')
    conn.execute('''CREATE TABLE IF NOT EXISTS olx_offers (
        id INTEGER PRIMARY KEY, olx_url TEXT NOT NULL, title TEXT NOT NULL,
        description TEXT, price_value INTEGER, price_currency TEXT DEFAULT 'PLN',
        price_label TEXT, price_negotiable INTEGER DEFAULT 0, price_arranged INTEGER DEFAULT 0,
        state TEXT, category_id INTEGER, category_type TEXT,
        city_id INTEGER, city_name TEXT, city_normalized TEXT,
        region_id INTEGER, region_name TEXT,
        lat REAL, lon REAL, map_radius REAL,
        user_id INTEGER, user_name TEXT, user_business INTEGER DEFAULT 0,
        user_online INTEGER DEFAULT 0, user_last_seen TEXT,
        has_phone INTEGER DEFAULT 0, has_chat INTEGER DEFAULT 0,
        delivery_active INTEGER DEFAULT 0, delivery_mode TEXT,
        photo_count INTEGER DEFAULT 0, thumbnail_url TEXT,
        promotion_top INTEGER DEFAULT 0, promotion_urgent INTEGER DEFAULT 0,
        promotion_highlighted INTEGER DEFAULT 0,
        status TEXT DEFAULT 'active', created_time TEXT, valid_to_time TEXT,
        last_refresh_time TEXT, first_seen_at TEXT NOT NULL,
        last_seen_at TEXT NOT NULL, scan_batch_id TEXT,
        raw_params_json TEXT, raw_delivery_json TEXT)''')
    conn.execute('''CREATE TABLE IF NOT EXISTS olx_offer_photos (
        id INTEGER PRIMARY KEY AUTOINCREMENT, offer_id INTEGER NOT NULL,
        photo_id INTEGER, cdn_url TEXT NOT NULL,
        width INTEGER, height INTEGER, rotation INTEGER DEFAULT 0,
        sort_order INTEGER DEFAULT 0,
        FOREIGN KEY (offer_id) REFERENCES olx_offers(id) ON DELETE CASCADE)''')
    conn.execute('''CREATE TABLE IF NOT EXISTS olx_offer_tags (
        id INTEGER PRIMARY KEY AUTOINCREMENT, offer_id INTEGER NOT NULL,
        tag TEXT NOT NULL, tag_source TEXT NOT NULL DEFAULT 'title',
        confidence REAL DEFAULT 1.0,
        FOREIGN KEY (offer_id) REFERENCES olx_offers(id) ON DELETE CASCADE,
        CONSTRAINT olx_tag_unique UNIQUE (offer_id, tag, tag_source))''')
    conn.execute('''CREATE TABLE IF NOT EXISTS olx_scan_batches (
        id TEXT PRIMARY KEY, started_at TEXT NOT NULL, finished_at TEXT,
        status TEXT NOT NULL DEFAULT 'running', category_id INTEGER,
        city_id INTEGER, distance_km INTEGER, total_available INTEGER,
        offers_scanned INTEGER DEFAULT 0, offers_new INTEGER DEFAULT 0,
        offers_updated INTEGER DEFAULT 0, offers_expired INTEGER DEFAULT 0,
 error_message TEXT, notebook_run_url TEXT, scraper_version TEXT DEFAULT '2.0')''')
    conn.execute('''CREATE TABLE IF NOT EXISTS olx_price_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT, offer_id INTEGER NOT NULL,
        price_value INTEGER, price_label TEXT, observed_at TEXT NOT NULL,
        scan_batch_id TEXT,
        FOREIGN KEY (offer_id) REFERENCES olx_offers(id) ON DELETE CASCADE)''')
    conn.execute('''CREATE TABLE IF NOT EXISTS olx_offer_parts_xref (
        id INTEGER PRIMARY KEY AUTOINCREMENT, offer_id INTEGER NOT NULL,
        master_part_id INTEGER, matched_part_name TEXT,
        match_method TEXT NOT NULL DEFAULT 'keyword', match_confidence REAL DEFAULT 0.5,
        created_at TEXT NOT NULL,
        FOREIGN KEY (offer_id) REFERENCES olx_offers(id) ON DELETE CASCADE,
        CONSTRAINT olx_xref_unique UNIQUE (offer_id, master_part_id, match_method))''')
    for idx_sql in [
        'CREATE INDEX IF NOT EXISTS idx_offers_cat_city ON olx_offers(category_id,city_id)',
        'CREATE INDEX IF NOT EXISTS idx_offers_price ON olx_offers(price_value,category_id)',
        'CREATE INDEX IF NOT EXISTS idx_offers_region ON olx_offers(region_id,category_id)',
        'CREATE INDEX IF NOT EXISTS idx_offers_status ON olx_offers(status,last_seen_at)',
        'CREATE INDEX IF NOT EXISTS idx_offers_user ON olx_offers(user_id)',
        'CREATE INDEX IF NOT EXISTS idx_offers_title ON olx_offers(title)',
        'CREATE INDEX IF NOT EXISTS idx_offers_created ON olx_offers(created_time DESC)',
        'CREATE INDEX IF NOT EXISTS idx_tags_tag ON olx_offer_tags(tag,confidence DESC)',
        'CREATE INDEX IF NOT EXISTS idx_price_hist ON olx_price_history(offer_id,observed_at DESC)',
        'CREATE INDEX IF NOT EXISTS idx_photos_offer ON olx_offer_photos(offer_id,sort_order)',
        'CREATE INDEX IF NOT EXISTS idx_xref_offer ON olx_offer_parts_xref(offer_id)',
        'CREATE INDEX IF NOT EXISTS idx_xref_part ON olx_offer_parts_xref(master_part_id,matched_part_name)',
    ]:
        conn.execute(idx_sql)
    conn.commit()
    return conn

db = init_db(DB_PATH)
tables = [r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()]
print(f'✅ DB ready. Tables: {tables}')


In [ ]:
# Core scraper functions
def extract_price(params):
    for p in params:
        if p.get('key') == 'price':
            v = p.get('value', {})
            return {'price_value': v.get('value'), 'price_currency': v.get('currency','PLN'),
                    'price_label': v.get('label',''), 'price_negotiable': 1 if v.get('negotiable') else 0,
                    'price_arranged': 1 if v.get('arranged') else 0}
    return {'price_value': None, 'price_currency': 'PLN', 'price_label': '', 'price_negotiable': 0, 'price_arranged': 0}

def extract_state(params):
    for p in params:
        if p.get('key') == 'state':
            return p.get('value',{}).get('key','')
    return ''

def insert_offer(conn, offer, now_iso, batch_id):
    oid = offer['id']
    pi = extract_price(offer.get('params',[]))
    loc = offer.get('location',{}) or {}
    city = loc.get('city',{}) or {}; region = loc.get('region',{}) or {}
    m = offer.get('map',{}) or {}; u = offer.get('user',{}) or {}
    ct = offer.get('contact',{}) or {}; dv = offer.get('delivery',{}) or {}
    pr = offer.get('promotion',{}) or {}; ph = offer.get('photos',[]) or []
    cat = offer.get('category',{}) or {}
    existing = conn.execute('SELECT id,price_value FROM olx_offers WHERE id=?',(oid,)).fetchone()
    row = {'id':oid, 'olx_url':offer.get('url',''), 'title':offer.get('title',''),
           'description':offer.get('description',''), 'state':extract_state(offer.get('params',[])),
           'category_id':cat.get('id'), 'category_type':cat.get('type'),
           'city_id':city.get('id'), 'city_name':city.get('name'),
           'city_normalized':city.get('normalized_name'),
           'region_id':region.get('id'), 'region_name':region.get('name'),
           'lat':m.get('lat'), 'lon':m.get('lon'), 'map_radius':m.get('radius'),
           'user_id':u.get('id'), 'user_name':u.get('name'),
           'user_business':1 if (u.get('business') or offer.get('business')) else 0,
           'user_online':1 if u.get('is_online') else 0,
           'user_last_seen':u.get('last_seen'),
           'has_phone':1 if ct.get('phone') else 0, 'has_chat':1 if ct.get('chat') else 0,
           'delivery_active':1 if dv.get('rock',{}).get('active') else 0,
           'delivery_mode':dv.get('rock',{}).get('mode'),
           'photo_count':len(ph), 'thumbnail_url':ph[0].get('link') if ph else None,
           'promotion_top':1 if pr.get('top_ad') else 0,
           'promotion_urgent':1 if pr.get('urgent') else 0,
           'promotion_highlighted':1 if pr.get('highlighted') else 0,
           'status':offer.get('status','active'),
           'created_time':offer.get('created_time'),
           'valid_to_time':offer.get('valid_to_time'),
           'last_refresh_time':offer.get('last_refresh_time'),
           'last_seen_at':now_iso, 'scan_batch_id':batch_id,
           'raw_params_json':json.dumps(offer.get('params',[]),ensure_ascii=False),
           'raw_delivery_json':json.dumps(dv,ensure_ascii=False), **pi}
    if existing:
        sets = ', '.join(f'{k}=?' for k in row if k!='id')
        vals = [row[k] for k in row if k!='id'] + [oid]
        conn.execute(f'UPDATE olx_offers SET {sets} WHERE id=?', vals)
        if existing['price_value'] != row.get('price_value') and row.get('price_value') is not None:
            conn.execute('INSERT INTO olx_price_history (offer_id,price_value,price_label,observed_at,scan_batch_id) VALUES (?,?,?,?,?)',
                (oid, row['price_value'], row['price_label'], now_iso, batch_id))
        return {'new':False, 'updated':True}
    else:
        row['first_seen_at'] = now_iso
        cols = ', '.join(row.keys()); phs = ', '.join('?' for _ in row)
        conn.execute(f'INSERT INTO olx_offers ({cols}) VALUES ({phs})', list(row.values()))
        for i,p in enumerate(ph):
            conn.execute('INSERT OR IGNORE INTO olx_offer_photos (offer_id,photo_id,cdn_url,width,height,rotation,sort_order) VALUES (?,?,?,?,?,?,?)',
                (oid,p.get('id'),p.get('link'),p.get('width'),p.get('height'),p.get('rotation',0),i))
        return {'new':True, 'updated':False}


def olx_preflight_host(timeout=5.0):
    """Verify Kaggle/local runtime can reach OLX before burning API attempts."""
    try:
        with socket.create_connection(('www.olx.pl', 443), timeout=timeout):
            return True
    except OSError as e:
        print(f'  🚫 OLX preflight failed: {e}')
        return False

def warmup_olx_session(client):
    """Open the public OLX page first so the API request carries a browser-like session."""
    try:
        resp = client.get('https://www.olx.pl/', headers=OLX_BROWSER_HEADERS, timeout=15.0)
        if resp.status_code in OLX_BLOCK_STATUS_CODES:
            print(f'  🚫 OLX warmup blocked: HTTP {resp.status_code}')
            return False
        print(f'  ✅ OLX warmup: HTTP {resp.status_code}')
        return True
    except httpx.HTTPError as e:
        print(f'  ⚠️ OLX warmup failed: {e}')
        return False

def fetch_olx_json(client, params, label):
    """Fetch OLX JSON with retry handling for Kaggle network flakiness and 403/429 blocks."""
    last_error = None
    for attempt in range(1, MAX_FETCH_ATTEMPTS + 1):
        try:
            resp = client.get(OLX_BASE_URL, params=params, headers=OLX_HEADERS, timeout=20.0)
            if resp.status_code in OLX_BLOCK_STATUS_CODES:
                last_error = f'HTTP {resp.status_code}'
                print(f'  🚫 {label}: OLX block {last_error} (attempt {attempt}/{MAX_FETCH_ATTEMPTS})')
                warmup_olx_session(client)
            else:
                resp.raise_for_status()
                try:
                    return resp.json()
                except ValueError:
                    snippet = resp.text[:160].replace('\n', ' ')
                    last_error = f'non-JSON response: {snippet!r}'
                    print(f'  ❌ {label}: {last_error}')
        except httpx.HTTPError as e:
            last_error = str(e)
            print(f'  ❌ {label}: request failed (attempt {attempt}/{MAX_FETCH_ATTEMPTS}): {e}')
        if attempt < MAX_FETCH_ATTEMPTS:
            time.sleep(REQUEST_DELAY_SEC * attempt)
    raise RuntimeError(f'OLX fetch failed after {MAX_FETCH_ATTEMPTS} attempts: {last_error}')

def scrape_offers(client, conn, category_id, city_id, distance, batch_id,
                  price_max=None, price_free_only=False, extra_label=''):
    now_iso = datetime.now(timezone.utc).isoformat()
    stats = {'scanned':0,'new':0,'updated':0,'errors':0,'pages':0}
    label = extra_label or f'cat={category_id}'
    offset = 0; total_available = None
    while True:
        params = {'category_id':category_id,'city_id':city_id,'distance':distance,
                  'limit':PAGE_LIMIT,'offset':offset}
        if price_free_only:
            params['search[filter_enum_price][0]'] = 'free'
        if price_max:
            params['search[filter_float_price:to]'] = str(price_max)
        try:
            data = fetch_olx_json(client, params, label=f'{label} offset={offset}')
        except Exception as e:
            print(f'  ❌ {label} offset={offset}: {e}')
            stats['errors'] += 1
            if stats['errors'] >= 5: break
            time.sleep(3); continue
        offers = data.get('data',[])
        if not offers: break
        if total_available is None:
            total_available = data.get('metadata',{}).get('visible_total_count',0)
            print(f'  📊 {label}: {total_available} ofert dostępnych')
        for offer in offers:
            r = insert_offer(conn, offer, now_iso, batch_id)
            if r['new']: stats['new'] += 1
            elif r['updated']: stats['updated'] += 1
            stats['scanned'] += 1
        stats['pages'] += 1
        conn.commit()
        if not data.get('links',{}).get('next',{}).get('href'): break
        offset += len(offers)
        if stats['pages'] >= MAX_PAGES: break
        time.sleep(REQUEST_DELAY_SEC)
    batch_key = f'{batch_id}-{label.replace(chr(32),"-")}'
    conn.execute('INSERT OR REPLACE INTO olx_scan_batches (id,started_at,status,category_id,city_id,distance_km,total_available,offers_scanned,offers_new,offers_updated,scraper_version) VALUES (?,?,?,?,?,?,?,?,?,?,?)',
        (batch_key, now_iso, 'done', category_id, city_id, distance, total_available or 0, stats['scanned'], stats['new'], stats['updated'], SCRAPER_VERSION))
    conn.commit()
    stats['total_available'] = total_available
    return stats

print('✅ Scraper functions ready')


In [ ]:
# ═══ FAZA 1: Run all scans ═══
client = httpx.Client(follow_redirects=True, headers=OLX_HEADERS)
if not olx_preflight_host():
    raise RuntimeError('Brak polaczenia z www.olx.pl. W Kaggle wlacz Settings -> Internet -> On albo uruchom lokalnie.')
warmup_olx_session(client)
all_stats = {}

# 1) Watchlisty Krzyska
print('🏠🔧🧊 WATCHLISTY KRZYSKA')
for wl_name, wl_cfg in WATCHLISTS.items():
    for cat_id in wl_cfg['categories']:
        label = f"{wl_cfg['label']}-cat{cat_id}"
        print(f'\n🔍 Scanning: {label}')
        s = scrape_offers(client, db, cat_id, wl_cfg['city_id'], wl_cfg['distance'], SCAN_BATCH_ID,
                          price_max=wl_cfg.get('max_price'),
                          price_free_only=wl_cfg.get('price_free',False),
                          extra_label=label)
        all_stats[label] = s
        print(f'  ✅ {label}: {s["scanned"]} scanned, {s["new"]} new, {s["updated"]} updated')

# 2) Oddam za darmo — full scan
print('\n🆓 ODDAM ZA DARMO — FULL SCAN')
for city_name, city_id, dist in FREE_SCAN_TARGETS:
    label = f'free-{city_name}'
    print(f'\n🔍 Scanning: {label}')
    s = scrape_offers(client, db, CAT_FREE, city_id, dist, SCAN_BATCH_ID, extra_label=label)
    all_stats[label] = s
    print(f'  ✅ {label}: {s["scanned"]} scanned, {s["new"]} new, {s["updated"]} updated')

client.close()
print(f'\n═══ FAZA 1 COMPLETE ═══')
total_scanned = sum(s['scanned'] for s in all_stats.values())
total_new = sum(s['new'] for s in all_stats.values())
total_updated = sum(s['updated'] for s in all_stats.values())
print(f'Total: {total_scanned} scanned, {total_new} new, {total_updated} updated')


In [ ]:
# ═══ FAZA 2: Tag + categorize offers ═══

# Simple keyword-based tagging (no AI needed)
TAG_RULES = {
    'laptop': ['laptop','notebook','ultrabook'],
    'telewizor': ['telewizor','tv ','oled','qled'],
    'telefon': ['telefon','smartfon','iphone','samsung galaxy','xiaomi'],
    'pralka': ['pralka','washing'],
    'lodówka': ['lodówka','chłodziarka','fridge','chłodziarka'],
    'zmywarka': ['zmywarka','dishwasher'],
    'piekarnik': ['piekarnik','kuchnia','indukcja','piec'],
    'meble': ['szafa','łóżko','kanapa','sofa','komoda','regał','biurko','krzesło','stół'],
    'rower': ['rower','bicycle','bike'],
    'narzędzia': ['wiertarka','młotek','szlifierka','saw','wkrętarka'],
    'kino_domowe': ['soundbar','głośnik','amplituner','receiver','home cinema'],
    'drukarka': ['drukarka','printer','laserjet'],
    'klimatyzacja': ['klimatyzator','klima','ac '],
    'dom': ['dom','mieszkanie','kawalerka','apartament','blok'],
    'działka': ['działka','grunt','teren','pole'],
    'samochód': ['samochód','auto','vw','audi','bmw','opel','ford','toyota','skoda'],
}

def auto_tag(conn):
    offers = conn.execute('SELECT id, title, description FROM olx_offers WHERE id NOT IN (SELECT DISTINCT offer_id FROM olx_offer_tags)').fetchall()
    tagged = 0
    for o in offers:
        text = f'{o["title"]} {o["description"] or ""}'.lower()
        for tag, keywords in TAG_RULES.items():
            for kw in keywords:
                if kw in text:
                    conn.execute('INSERT OR IGNORE INTO olx_offer_tags (offer_id,tag,tag_source,confidence) VALUES (?,?,?,?)',
                        (o['id'], tag, 'keyword', 0.8))
                    tagged += 1
                    break
    conn.commit()
    return tagged

tagged = auto_tag(db)
print(f'✅ Tagged {tagged} offer-tag pairs')

# Watchlist filtered views
for wl_name, wl_cfg in WATCHLISTS.items():
    cats = wl_cfg['categories']
    cat_placeholders = ','.join(['?']*len(cats))
    query = f'SELECT id, title, price_label, city_name, created_time FROM olx_offers WHERE category_id IN ({cat_placeholders}) AND city_id=? AND status="active" ORDER BY created_time DESC LIMIT 20'
    params = list(cats) + [wl_cfg['city_id']]
    if wl_cfg.get('max_price'):
        query = query.replace('ORDER BY', 'AND price_value <= ? ORDER BY')
        params.insert(-1, wl_cfg['max_price'])
    rows = db.execute(query, params).fetchall()
    print(f'\n{wl_cfg["label"]} ({len(rows)} ofert):')
    for r in rows[:5]:
        print(f'  • {r["title"][:60]} | {r["price_label"] or "—"} | {r["city_name"]}')


---
# 🧠 FAZA 2B: AI Tagging z Gemma 4 (OPCJONALNIE)

**Domyślnie WYŁĄCZONE.** Ustaw `GEMMA_ENABLED = True` aby włączyć.

## Wymagania:
- Kaggle notebook z **GPU T4/P100** włączonym (Settings → Accelerator → GPU)
- `kaggle_secrets` z `GEMMA_TOKEN` (Kaggle API key do modeli)

## Dostępne modele na Kaggle P100 (16GB VRAM):
| Model | Parametry | VRAM | Context | Multimodal | Rekomendacja |
|-------|-----------|------|---------|------------|-------------|
| Gemma 4 E4B-it | 4.5B eff | ~8GB | 128K | Text+Image+Audio | ✅ Szybki, oszczędny |
| Gemma 4 26B A4B-it (4bit) | 3.8B active | ~14GB | 256K | Text+Image | 🏆 Najlepszy na P100 |
| Gemma 4 31B Dense (4bit) | 30.7B | ~18GB | 256K | Text+Image | ⚠️ Na granicy VRAM |

**Rekomendacja:** `gemma-4-26b-a4b-it` z 4-bit quantization — inferencja jak 4B model, jakość jak 27B.


In [ ]:
# ═══ FAZA 2B: Gemma 4 AI Tagging (OPCJONALNIE — domyślnie wyłączone) ═══
# Ustaw GEMMA_ENABLED = True aby włączyć AI tagging zamiast keyword-only

GEMMA_ENABLED = False  # ← Flip to True when ready

# Model selection — zmień na inny wariant jeśli potrzeba
# Opcje: 'gemma-4-e4b-it', 'gemma-4-26b-a4b-it', 'gemma-4-31b-it'
GEMMA_MODEL_VARIANT = 'gemma-4-26b-a4b-it'  # 🏆 Best on P100
GEMMA_QUANTIZATION = '4bit'  # '4bit' dla 26B/31B, 'full' dla E4B
GEMMA_KAGGLE_MODEL_HANDLE = f'google/gemma-4/transformers/gemma-4-26b-a4b-it'

# ──────────────────────────────────────────
# NIE EDYTUJ PONIŻEJ — infrastruktura gotowa
# ──────────────────────────────────────────

def load_gemma_model(model_handle, quantization='4bit'):
    """Load Gemma 4 model from Kaggle Models hub.
    
    Requires:
    - GPU accelerator enabled in notebook settings
    - Kaggle secret: KAGGLE_KEY (auto-available on Kaggle)
    - kagglehub installed: !pip install kagglehub
    
    Returns: (model, tokenizer) tuple
    """
    import kagglehub
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch
    
    print(f'📥 Downloading {model_handle}...')
    model_path = kagglehub.model_download(model_handle)
    print(f'   Path: {model_path}')
    
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    if quantization == '4bit':
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            quantization_config=bnb_config,
            device_map='auto',
            torch_dtype=torch.bfloat16,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map='auto',
            torch_dtype=torch.bfloat16,
        )
    
    model.eval()
    vram_mb = torch.cuda.memory_allocated() / 1024**2
    print(f'✅ Model loaded. VRAM: {vram_mb:.0f} MB')
    return model, tokenizer


def gemma_tag_offer(model, tokenizer, title, description=''):
    """Tag single offer using Gemma 4. Returns dict with tags + summary.
    
    Tags: category, condition, brand (if detectable), value_score (1-10),
          summary (1-sentence PL), recommended (bool).
    """
    import torch
    
    prompt = f"""<start_of_turn>user
Jesteś asystentem do tagowania ofert OLX. Przeanalizuj ofertę i zwróć TYLKO JSON.

Oferta: {title}
Opis: {(description or 'brak')[:500]}

Zwróć JSON z kluczami:
- "category": główna kategoria (elektronika/meble/agd/odzież/rower/narzędzia/dom/działka/samochód/inne)
- "subcategory": podkategoria (np. laptop, pralka, szafa)
- "condition": nowy/używany/do_renowacji/parts
- "brand": marka jeśli wykryta lub null
- "value_score": 1-10 (10=super okazja darmowa)
- "summary": jedno zdanie po polsku
- "recommended": true/false (czy warto zainteresować się tą ofertą)

JSON:<end_of_turn>
<start_of_turn>model
{{"""
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Parse JSON from response
    try:
        # Find JSON in response
        json_str = '{' + response.split('{', 1)[1] if '{' in response else response
        json_str = json_str.rsplit('}', 1)[0] + '}' if '}' in json_str else json_str
        result = json.loads(json_str)
    except (json.JSONDecodeError, IndexError):
        result = {'category': 'inne', 'subcategory': '', 'condition': 'nieznany',
                  'brand': None, 'value_score': 5, 'summary': response[:100],
                  'recommended': False, 'parse_error': True}
    
    return result


def gemma_batch_tag(conn, model, tokenizer, limit=100):
    """Tag untagged offers using Gemma 4. Batch processing with GPU.
    
    Args:
        conn: sqlite3 connection
        model: loaded Gemma model
        tokenizer: loaded tokenizer
        limit: max offers to process (GPU time is precious!)
    
    Returns: dict with stats
    """
    import torch
    
    offers = conn.execute(
        'SELECT id, title, description FROM olx_offers '
        'WHERE id NOT IN (SELECT DISTINCT offer_id FROM olx_offer_tags WHERE tag_source="gemma") '
        'ORDER BY created_time DESC LIMIT ?',
        (limit,)
    ).fetchall()
    
    stats = {'processed': 0, 'tagged': 0, 'errors': 0}
    
    for i, o in enumerate(offers):
        try:
            result = gemma_tag_offer(model, tokenizer, o['title'], o['description'])
            
            # Insert tags from Gemma analysis
            tags_to_insert = [
                (o['id'], result.get('category', 'inne'), 'gemma', 0.95),
                (o['id'], result.get('subcategory', ''), 'gemma_sub', 0.9),
                (o['id'], result.get('condition', ''), 'gemma_condition', 0.85),
            ]
            if result.get('brand'):
                tags_to_insert.append((o['id'], result['brand'].lower(), 'gemma_brand', 0.8))
            
            for offer_id, tag, source, conf in tags_to_insert:
                if tag:  # skip empty tags
                    conn.execute(
                        'INSERT OR IGNORE INTO olx_offer_tags (offer_id,tag,tag_source,confidence) VALUES (?,?,?,?)',
                        (offer_id, tag, source, conf)
                    )
            
            # Store value_score + summary in description field as metadata
            # (we add a special tag for value_score)
            vs = result.get('value_score', 5)
            conn.execute(
                'INSERT OR IGNORE INTO olx_offer_tags (offer_id,tag,tag_source,confidence) VALUES (?,?,?,?)',
                (o['id'], f'value_{vs}', 'gemma_score', vs / 10.0)
            )
            
            if result.get('recommended'):
                conn.execute(
                    'INSERT OR IGNORE INTO olx_offer_tags (offer_id,tag,tag_source,confidence) VALUES (?,?,?,?)',
                    (o['id'], 'recommended', 'gemma_rec', 0.9)
                )
            
            stats['tagged'] += 1
            
        except Exception as e:
            stats['errors'] += 1
            if stats['errors'] >= 10:
                print(f'  ❌ Too many errors, stopping. Last: {e}')
                break
        
        stats['processed'] += 1
        if (i + 1) % 10 == 0:
            conn.commit()
            vram = torch.cuda.memory_allocated() / 1024**2
            print(f'  📊 {i+1}/{len(offers)} processed | {stats["tagged"]} tagged | VRAM: {vram:.0f}MB')
    
    conn.commit()
    return stats


# ═══ Run Gemma tagging if enabled ═══
if GEMMA_ENABLED:
    print(f'🧠 Gemma 4 AI tagging ENABLED — model: {GEMMA_MODEL_VARIANT}')
    print(f'   Quantization: {GEMMA_QUANTIZATION}')
    print(f'   Loading model... (this takes ~2-5 min on first run)')
    
    model, tokenizer = load_gemma_model(GEMMA_KAGGLE_MODEL_HANDLE, GEMMA_QUANTIZATION)
    
    # Process untagged offers (limit to save GPU hours)
    GEMMA_BATCH_LIMIT = 200  # adjust based on 30h/week GPU quota
    gemma_stats = gemma_batch_tag(db, model, tokenizer, limit=GEMMA_BATCH_LIMIT)
    
    print(f'\n✅ Gemma tagging complete:')
    print(f'   Processed: {gemma_stats["processed"]}')
    print(f'   Tagged: {gemma_stats["tagged"]}')
    print(f'   Errors: {gemma_stats["errors"]}')
    
    # Cleanup GPU memory
    import torch
    del model, tokenizer
    torch.cuda.empty_cache()
    print('🧹 GPU memory freed')
else:
    print('⏭️ Gemma AI tagging SKIPPED (GEMMA_ENABLED=False)')
    print('   Aby włączyć: ustaw GEMMA_ENABLED = True w pierwszej linijce tej komórki')
    print('   Wymaga: GPU accelerator ON + kagglehub + transformers + bitsandbytes')


In [ ]:
# ═══ FAZA 3: Export + GitHub push ═══

# 1) Export JSONL (one file per watchlist + one for free items)
def export_watchlist_jsonl(conn, wl_name, wl_cfg, outdir):
    cats = wl_cfg['categories']
    cat_ph = ','.join(['?']*len(cats))
    q = f'SELECT * FROM olx_offers WHERE category_id IN ({cat_ph}) AND city_id=? AND status="active" ORDER BY created_time DESC'
    params = list(cats) + [wl_cfg['city_id']]
    if wl_cfg.get('max_price'):
        q = q.replace('ORDER BY', 'AND price_value <= ? ORDER BY')
        params.insert(-1, wl_cfg['max_price'])
    rows = conn.execute(q, params).fetchall()
    path = outdir / f'{wl_name}_{datetime.now().strftime("%Y%m%d")}.jsonl'
    with open(path,'w',encoding='utf-8') as f:
        for r in rows:
            d = dict(r)
            d.pop('raw_params_json',None); d.pop('raw_delivery_json',None)
            f.write(json.dumps(d, ensure_ascii=False, default=str) + '\n')
    return len(rows), path

for wl_name, wl_cfg in WATCHLISTS.items():
    cnt, p = export_watchlist_jsonl(db, wl_name, wl_cfg, WATCHLIST_DIR)
    print(f'✅ {wl_cfg["label"]}: {cnt} offers → {p.name}')

# 2) Free items export
free_rows = db.execute('SELECT * FROM olx_offers WHERE category_id=? AND status="active" ORDER BY created_time DESC', (CAT_FREE,)).fetchall()
free_path = WATCHLIST_DIR / f'oddam_za_darmo_{datetime.now().strftime("%Y%m%d")}.jsonl'
with open(free_path,'w',encoding='utf-8') as f:
    for r in free_rows:
        d = dict(r); d.pop('raw_params_json',None); d.pop('raw_delivery_json',None)
        f.write(json.dumps(d, ensure_ascii=False, default=str) + '\n')
print(f'✅ Oddam za darmo: {len(free_rows)} offers → {free_path.name}')

# 3) D1 bulk exports (JSONL + SQL import file)
D1_EXPORT_TABLES = [
    'olx_offers', 'olx_offer_photos', 'olx_offer_tags',
    'olx_scan_batches', 'olx_price_history', 'olx_offer_parts_xref',
]

def sql_literal(value):
    if value is None:
        return 'NULL'
    if isinstance(value, (int, float)):
        return str(value)
    s = str(value).replace("'", "''").replace(chr(10), ' ').replace(chr(13), '')
    return "'" + s + "'"

def export_to_jsonl(conn, output_path):
    count = 0
    with open(output_path, 'w', encoding='utf-8') as f:
        for table in D1_EXPORT_TABLES:
            for row in conn.execute(f'SELECT * FROM {table}'):
                f.write(json.dumps({'table': table, 'row': dict(row)}, ensure_ascii=False, default=str) + '\n')
                count += 1
    return count

def export_to_sql(conn, output_path):
    count = 0
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('-- OLX Parts Market D1 Import\n')
        f.write(f'-- Batch: {SCAN_BATCH_ID}\n')
        f.write(f'-- Generated: {datetime.now(timezone.utc).isoformat()}\n\n')
        for table in D1_EXPORT_TABLES:
            rows = conn.execute(f'SELECT * FROM {table}').fetchall()
            if not rows:
                continue
            f.write(f'-- Table: {table}\n')
            for row in rows:
                data = dict(row)
                cols = [k for k, v in data.items() if v is not None]
                col_sql = ', '.join(cols)
                val_sql = ', '.join(sql_literal(data[c]) for c in cols)
                f.write(f'INSERT OR REPLACE INTO {table} ({col_sql}) VALUES ({val_sql});\n')
                count += 1
            f.write('\n')
    return count

jsonl_count = export_to_jsonl(db, JSONL_PATH)
sql_count = export_to_sql(db, SQL_PATH)
print(f'✅ D1 JSONL: {jsonl_count} rows → {JSONL_PATH.name}')
print(f'✅ D1 SQL: {sql_count} rows → {SQL_PATH.name}')

# 4) Generate report
report = [f'# OLX Scan Report — {datetime.now().strftime("%Y-%m-%d %H:%M")}\n']
report.append(f'Batch: {SCAN_BATCH_ID}\n')
report.append('## Watchlisty\n')
for wl_name, wl_cfg in WATCHLISTS.items():
    cats = wl_cfg['categories']
    cat_ph = ','.join(['?']*len(cats))
    cnt = db.execute(f'SELECT COUNT(*) FROM olx_offers WHERE category_id IN ({cat_ph}) AND city_id=? AND status="active"',
        list(cats)+[wl_cfg['city_id']]).fetchone()[0]
    report.append(f'- {wl_cfg["label"]}: {cnt} ofert\n')
report.append(f'\n## Oddam za darmo\n')
report.append(f'- Total: {len(free_rows)} ofert\n')
report.append(f'\n## DB Stats\n')
total_offers = db.execute('SELECT COUNT(*) FROM olx_offers').fetchone()[0]
total_tags = db.execute('SELECT COUNT(*) FROM olx_offer_tags').fetchone()[0]
report.append(f'- Offers: {total_offers}\n')
report.append(f'- Tags: {total_tags}\n')
report.append(f'- D1 JSONL rows: {jsonl_count} ({JSONL_PATH.name})\n')
report.append(f'- D1 SQL rows: {sql_count} ({SQL_PATH.name})\n')
with open(REPORT_PATH,'w',encoding='utf-8') as f:
    f.writelines(report)
print(f'✅ Report: {REPORT_PATH}')

# 5) GitHub push (if PAT configured)
if GITHUB_PAT:
    print('\n🚀 Pushing to GitHub...')
    import base64
    api_url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{GITHUB_DATA_PATH}'
    headers = {'Authorization': f'token {GITHUB_PAT}', 'Accept': 'application/vnd.github.v3+json'}
    for fpath in WATCHLIST_DIR.glob('*.jsonl'):
        with open(fpath,'rb') as f:
            content = base64.b64encode(f.read()).decode()
        check = httpx.get(f'{api_url}/{fpath.name}', headers=headers)
        sha = check.json().get('sha') if check.status_code == 200 else None
        payload = {'message': f'olx scan: {fpath.name}', 'content': content, 'branch': GITHUB_BRANCH}
        if sha: payload['sha'] = sha
        r = httpx.put(f'{api_url}/{fpath.name}', headers=headers, json=payload)
        print(f'  {fpath.name}: {r.status_code}')
else:
    print('⚠️ No GITHUB_PAT — files saved locally only')

print(f'\n═══ FAZA 3 COMPLETE ═══')


In [ ]:
# ═══ Summary Dashboard ═══
total_offers = db.execute('SELECT COUNT(*) FROM olx_offers').fetchone()[0]
total_photos = db.execute('SELECT COUNT(*) FROM olx_offer_photos').fetchone()[0]
total_tags = db.execute('SELECT COUNT(*) FROM olx_offer_tags').fetchone()[0]
total_batches = db.execute('SELECT COUNT(*) FROM olx_scan_batches').fetchone()[0]
free_count = db.execute('SELECT COUNT(*) FROM olx_offers WHERE category_id=?',(CAT_FREE,)).fetchone()[0]
active_count = db.execute('SELECT COUNT(*) FROM olx_offers WHERE status="active"').fetchone()[0]

print('╔══════════════════════════════════════╗')
print('║   OLX MARKET SCANNER — DASHBOARD     ║')
print('╠══════════════════════════════════════╣')
print(f'║ Total offers:    {total_offers:>6}              ║')
print(f'║ Active offers:   {active_count:>6}              ║')
print(f'║ Free items:      {free_count:>6}              ║')
print(f'║ Photos stored:   {total_photos:>6}              ║')
print(f'║ Tags assigned:   {total_tags:>6}              ║')
print(f'║ Scan batches:    {total_batches:>6}              ║')
print('╠══════════════════════════════════════╣')
for wl_name, wl_cfg in WATCHLISTS.items():
    cats = wl_cfg['categories']
    cat_ph = ','.join(['?']*len(cats))
    cnt = db.execute(f'SELECT COUNT(*) FROM olx_offers WHERE category_id IN ({cat_ph}) AND city_id=? AND status="active"',
        list(cats)+[wl_cfg['city_id']]).fetchone()[0]
    print(f'║ {wl_cfg["label"]}: {cnt:>4}             ║')
print('╚══════════════════════════════════════╝')

# Top free items
print('\n🔥 Najnowsze darmowe oferty:')
top_free = db.execute('SELECT title, city_name, created_time, olx_url FROM olx_offers WHERE category_id=? AND status="active" ORDER BY created_time DESC LIMIT 10', (CAT_FREE,)).fetchall()
for r in top_free:
    print(f'  • {r["title"][:55]} | {r["city_name"]} | {r["olx_url"]}')
